In [1]:
import pandas as pd
import numpy as np
import tanalysis.traj_analysis as ta
from scipy.optimize import curve_fit
import matplotlib.pyplot as plt
import os

In [2]:
dirname = r"C:\Users\pcanaleta\Documents\Cellpose_segmentation\EXP.HD6.Chips\EXP.HD6.1.1.MatekChips_CXCL10\24h\Tracks\excel_tracks\CXCL10_Conc10.xlsx"
savedir = r"C:\Users\pcanaleta\Documents\Cellpose_segmentation\EXP.HD6.Chips\EXP.HD6.1.1.MatekChips_CXCL10\24h\Tracks\excel_tracks\Results"
filter_values = {'track_duration':(0,200000), 'mean_velocity':(0,100), 'total_distance':(0,100000)}
tracks, names = ta.crop_traj(dirname, filter_tracks=False, filter_values=filter_values)

In [7]:
params = ta.fit_APRW(tracks, names, fr'{savedir}\APRW')

In [11]:
ids = np.unique(params.index.get_level_values(0))
dt=1
for id in ids:
    track_params = params.loc[id]
    beta = 1/track_params['P']
    alpha = (track_params['S']**2)*beta
    map = max(0, 1-dt/track_params['P'])

ValueError: The truth value of a Series is ambiguous. Use a.empty, a.bool(), a.item(), a.any() or a.all().

In [ ]:
ids = np.unique(tracks.index.get_level_values(0))
for id in ids:
    track = tracks.loc[id]
    dt = np.unique(np.diff(track, axis=0)[:,0][0])[0]
    xyz = track.iloc[:,slice(1,None,1)].dropna()
    frames = xyz.index
    dxyz = np.diff(xyz, axis=0)
    U,S,V = np.linalg.svd(dxyz, full_matrices=False)
    xyzrot = xyz@V.T

    msdp0 = ta.ezmsd(xyzrot.iloc[:,0])
    msdnp0 = ta.ezmsd(xyzrot.iloc[:,1])
    if len(xyz.columns) == 3:
        msdnp0 += ta.ezmsd(xyzrot.iloc[:,2])
    
    t = np.arange(1,26,1)
    poptp, _ = curve_fit(ta.tPRW1D, frames[1:]*dt, msdp0[:], p0=(100,0.1,1), bounds=([0, 0, 0], [1000,100/dt,10]), method='trf', maxfev=10000)
    poptnp, _ = curve_fit(ta.tPRW2D, frames[1:]*dt, msdnp0[:], p0=(100,0.1,1), bounds=([0, 0, 0], [1000,100/dt,10]), method='trf', maxfev=10000)

    print(poptp, poptnp)
    index = pd.MultiIndex.from_product([ids,['p', 'np']])
    print(index)

    plt.figure()
    plt.plot(frames[1:]*dt, msdp0[:])
    plt.plot(frames[1:]*dt, ta.tPRW1D(frames[1:]*dt, *poptp))
    plt.plot(frames[1:]*dt, msdnp0[:])
    plt.plot(frames[1:]*dt, ta.tPRW2D(frames[1:]*dt, *poptnp))

    